# 🌍 Exercícios de Modelos Não Supervisionados de Machine Learning

**MBA em Data Science e Analytics (USP/ESALQ)**  
**Prof. Dr. Wilson Tarantin Junior**

---

Este notebook implementa uma **Análise Fatorial por Componentes Principais (PCA)** aplicada ao banco de dados PISA 2022, seguida de uma **análise de associação** entre o fator extraído e o grupo dos países.

Dataset: notas PISA — [OECD PISA Data Explorer](https://pisadataexplorer.oecd.org/ide/idepisa/report.aspx)

## ⚙️ Instalação dos Pacotes

Execute diretamente no **terminal/console** (não dentro do notebook):

```bash
pip install pandas numpy matplotlib seaborn plotly scipy statsmodels factor_analyzer
```

## 📥 Importação dos Pacotes

- `pandas` / `numpy` — manipulação e cálculo numérico
- `matplotlib` / `seaborn` — visualizações estáticas
- `factor_analyzer` — PCA e teste de Bartlett
- `scipy.stats.chi2_contingency` — teste qui-quadrado
- `statsmodels` — análise de resíduos padronizados ajustados
- `plotly` — visualização interativa do mapa de resíduos

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity
from scipy.stats import chi2_contingency
import statsmodels.api as sm
import plotly.graph_objects as go
import warnings
warnings.filterwarnings("ignore")

## 📂 Carregamento do Banco de Dados

- Arquivo: `notas_pisa.csv`
- Contém notas de países nas disciplinas de **Matemática**, **Leitura** e **Ciências** para os anos de 2018 e 2022
- Fonte: [OECD PISA Data Explorer](https://pisadataexplorer.oecd.org/ide/idepisa/report.aspx)

In [ ]:
pisa = pd.read_csv('notas_pisa.csv', delimiter=',')
pisa.info()

## 🗑️ Filtragem — Apenas Notas de 2022

- Removemos as colunas referentes a 2018 para focar na edição mais recente
- Reduz o número de variáveis e elimina potencial redundância entre ciclos

In [ ]:
pisa.drop(columns=['mathematics_2018', 'reading_2018', 'science_2018'], inplace=True)
pisa.head()

## 🔢 Conversão para Variáveis Numéricas

- As colunas de notas podem ter sido importadas como texto (`object`)
- Usamos `pd.to_numeric(..., errors='coerce')` para converter, forçando valores não numéricos a `NaN`

In [ ]:
pisa['mathematics_2022'] = pd.to_numeric(pisa['mathematics_2022'], errors='coerce')
pisa['reading_2022'] = pd.to_numeric(pisa['reading_2022'], errors='coerce')
pisa['science_2022'] = pd.to_numeric(pisa['science_2022'], errors='coerce')
pisa.dtypes

## 🧹 Remoção de Valores Ausentes

- Países sem nota registrada em 2022 geram entradas `NaN` após a conversão
- Removemos essas linhas para garantir que a PCA opere sobre dados completos

In [ ]:
pisa.dropna(inplace=True)
print(f'Linhas após remoção de NaN: {pisa.shape[0]}')

---
# 📐 Análise Fatorial por Componentes Principais (PCA)

## 🔀 Preparação do Dataset para a PCA

- Removemos as colunas identificadoras (`country`, `group`) — a PCA opera apenas sobre variáveis numéricas
- Resultado: `pisa_pca` contém somente as três notas de 2022

In [ ]:
pisa_pca = pisa.drop(columns=['country', 'group'])
pisa_pca.head()

## 🔗 Análise Gráfica das Correlações de Pearson

- Calcula a correlação de Pearson entre todas as variáveis numéricas
- Visualiza a matriz como heatmap para identificar relações lineares entre as disciplinas

In [ ]:
matriz_corr = pisa_pca.corr()

plt.figure(figsize=(12, 8), dpi=600)
sns.heatmap(matriz_corr, annot=True,
            cmap=plt.cm.Purples,
            annot_kws={'size': 12})
plt.title('Matriz de Correlações — PISA 2022')
plt.show()

### 💡 Interpretação

> Valores próximos de **+1** indicam correlação positiva forte entre as disciplinas.  
> Correlações elevadas entre `mathematics_2022`, `reading_2022` e `science_2022` justificam a existência de um **fator latente comum** — o desempenho educacional geral do país.  
> Isso valida a aplicação da PCA para extrair esse fator.

## 📊 Estatísticas Descritivas

- Visão geral da distribuição das notas: média, desvio-padrão, mínimo, máximo e quartis
- Permite identificar discrepâncias de escala antes da PCA

In [ ]:
pisa_pca.describe()

## 🧪 Teste de Esfericidade de Bartlett

- Verifica se as variáveis são **suficientemente correlacionadas** para justificar a PCA
- H₀: a matriz de correlações é uma matriz identidade (sem correlação entre variáveis)
- Critério: **p-valor < 0,05** → rejeita H₀ → PCA é adequada

In [ ]:
bartlett, p_value = calculate_bartlett_sphericity(pisa_pca)
print(f'Qui² Bartlett: {round(bartlett, 2)}')
print(f'p-valor: {round(p_value, 4)}')

### 💡 Interpretação

> **p-valor < 0,05** → rejeitamos H₀ → as variáveis possuím correlações significativas entre si.  
> Isso confirma que a PCA é **estatisticamente adequada** para este conjunto de dados.  
> Se o p-valor fosse ≥ 0,05, a PCA não seria recomendada, pois as variáveis seriam praticamente independentes.

## ⚙️ PCA Inicial — Todos os Fatores Possíveis

- Primeiro ajuste com `n_factors=3` (máximo possível para 3 variáveis) sem rotação
- Permite inspecionar todos os autovalores antes de aplicar o critério de seleção

In [ ]:
fa = FactorAnalyzer(n_factors=3, method='principal', rotation=None).fit(pisa_pca)
print('PCA inicial ajustada.')

## 🔢 Autovalores da PCA

- Os autovalores medem a variância explicada por cada componente
- A soma dos autovalores é igual ao número de variáveis
- Critério da raiz latente: **autovalor > 1** → componente retido

In [ ]:
autovalores = fa.get_eigenvalues()[0]
print('Autovalores:')
for i, av in enumerate(autovalores):
    print(f'  Componente {i+1}: {round(av, 4)}')

### 💡 Guia de Interpretação

> O **critério da raiz latente** (Kaiser) retém apenas fatores com autovalor **> 1**.  
> Isso significa que o fator deve explicar mais variância do que qualquer variável individual padronizada.  
> Com apenas 3 variáveis e correlações altas, é esperado que **1 único fator** seja suficiente.

## 🔄 PCA Final — Critério da Raiz Latente

- Reajustamos o modelo com `n_factors=1`, selecionado com base nos autovalores > 1
- Um único fator captura o **desempenho educacional geral** dos países

In [ ]:
fa = FactorAnalyzer(n_factors=1, method='principal', rotation=None).fit(pisa_pca)
print('PCA final ajustada com 1 fator.')

## 📊 Eigenvalues, Variâncias e Variâncias Acumuladas

- **Autovalor**: magnitude da variância capturada pelo fator
- **Variância (%)**: proporção da variância total explicada
- **Variância Acumulada (%)**: variância explicada acumulada até aquele fator

In [ ]:
autovalores_fatores = fa.get_factor_variance()

tabela_eigen = pd.DataFrame(autovalores_fatores)
tabela_eigen.columns = [f'Fator {i+1}' for i, v in enumerate(tabela_eigen.columns)]
tabela_eigen.index = ['Autovalor', 'Variância', 'Variância Acumulada']
tabela_eigen = tabela_eigen.T

print(tabela_eigen)

### 💡 Interpretação

> Quanto maior a **variância acumulada** do fator retido, mais fielmente ele representa o conjunto original.  
> Um único fator explicando **> 80%** da variância total indica que as três disciplinas do PISA são altamente colineares e convergem para uma dimensão latente comum — o **desempenho educacional geral** do país.

## 🔗 Cargas Fatoriais

- As **cargas fatoriais** medem a correlação entre cada variável e o fator extraído
- Valores absolutos próximos de **1** indicam forte associação com o fator
- Cargas **> 0,5** (em módulo) são geralmente consideradas relevantes

In [ ]:
cargas_fatoriais = fa.loadings_

tabela_cargas = pd.DataFrame(cargas_fatoriais)
tabela_cargas.columns = [f'Fator {i+1}' for i, v in enumerate(tabela_cargas.columns)]
tabela_cargas.index = pisa_pca.columns

print(tabela_cargas.round(4))

### 💡 Interpretação

> Cargas próximas de **1** confirmam que o fator extraído é fortemente determinado por todas as três disciplinas.  
> O sinal positivo indica que países com notas mais altas em matemática, leitura e ciências recebem **escores fatoriais maiores**.  
> Se alguma variável apresentasse carga baixa, ela contribuiria pouco para o fator e poderia ser excluída.

## 🧩 Comunalidades

- A **comunalidade** de uma variável é a proporção de sua variância explicada pelo fator retido
- Varia de 0 a 1: valores próximos de **1** indicam que a variável é bem representada
- Valores baixos sugerem que a variável possui variância específica não capturada

In [ ]:
comunalidades = fa.get_communalities()

tabela_comunalidades = pd.DataFrame(comunalidades)
tabela_comunalidades.columns = ['Comunalidades']
tabela_comunalidades.index = pisa_pca.columns

print(tabela_comunalidades.round(4))

### 💡 Interpretação

> Comunalidades **próximas de 1** → a variável é amplamente explicada pelo fator.  
> Comunalidades **abaixo de 0,5** → a variável tem variância específica significativa não capturada pelo modelo.  
> Comunalidades altas para todas as disciplinas reforçam a qualidade do fator único extraído.

## 🏗️ Extração do Fator para as Observações

- Calculamos o **escore fatorial** de cada país — sua posição no fator de desempenho educacional
- O fator é adicionado ao dataset original e o dataset é **ordenado** do maior para o menor escore

In [ ]:
fator = pd.DataFrame(fa.transform(pisa_pca))
fator.columns = ['fator_2022']

# Adicionando o fator ao banco de dados
pisa = pd.concat([pisa.reset_index(drop=True), fator], axis=1)

# Ordenando pelo fator (ranking dos países)
pisa.sort_values('fator_2022', ascending=False, inplace=True)
pisa.reset_index(drop=True, inplace=True)

pisa[['country', 'group', 'fator_2022']].head(10)

## 🏋️ Scores Fatoriais

- Os **scores** (pesos) indicam como cada variável contribui para o cálculo do escore de cada observação
- São os coeficientes da combinação linear que gera o fator
- Diferem das cargas fatoriais: cargas = correlação; scores = coeficientes de ponderação

In [ ]:
scores = fa.weights_

tabela_scores = pd.DataFrame(scores)
tabela_scores.columns = [f'Fator {i+1}' for i, v in enumerate(tabela_scores.columns)]
tabela_scores.index = pisa_pca.columns

print(tabela_scores.round(4))

---
# 🔍 Análise de Associação — Fator vs. Grupo de Países

## 🏷️ Categorização do Fator em Quartis

- Dividimos os países em **4 grupos** com base no escore fatorial (quartis)
- Categorias: `menores`, `médio_menor`, `médio_maior`, `maiores`
- Permite comparar o desempenho educacional com variáveis categóricas do dataset

In [ ]:
pisa['categoria'] = pd.qcut(
    pisa['fator_2022'],
    4,
    labels=['menores', 'médio_menor', 'médio_maior', 'maiores']
)

pisa['categoria'].value_counts().sort_index()

## 📋 Tabela de Contingência — Categoria vs. Grupo

- Cruza a **categoria do fator** (desempenho) com o **grupo do país** (ex.: OCDE vs. não-OCDE)
- Permite verificar visualmente se há padrão de associação entre as variáveis

In [ ]:
tabela = pd.crosstab(pisa['categoria'], pisa['group'])
print(tabela)

## 🧪 Teste Qui-Quadrado — Significância da Associação

- Testa se a associação entre categoria do fator e grupo dos países é **estatisticamente significativa**
- H₀: as variáveis são independentes (sem associação)
- Critério: **p-valor < 0,05** → rejeita H₀ → há associação

In [ ]:
teste_qui2 = chi2_contingency(tabela)

print(f'Estatística qui²: {round(teste_qui2[0], 2)}')
print(f'p-valor: {round(teste_qui2[1], 4)}')
print(f'Graus de liberdade: {teste_qui2[2]}')

### 💡 Interpretação

> **p-valor < 0,05** → rejeitamos H₀ → há associação estatisticamente significativa entre o desempenho educacional e o grupo do país.  
> Isso indica que países de determinados grupos (ex.: membros da OCDE) tendem a se concentrar nas categorias de maior desempenho no PISA 2022.

## 📉 Análise dos Resíduos Padronizados Ajustados

- Os **resíduos padronizados ajustados** medem o desvio de cada célula em relação à expectativa de independência
- Valores **> |1,96|** indicam associação significativa naquela célula específica (nível 5%)

In [ ]:
tab_cont = sm.stats.Table(tabela)

print('Resíduos Padronizados Ajustados:')
print(tab_cont.standardized_resids.round(4))

### 💡 Interpretação

> Resíduos **> +1,96**: frequência observada **maior** do que o esperado sob independência — associação positiva.  
> Resíduos **< −1,96**: frequência observada **menor** do que o esperado — associação negativa.  
> Resíduos entre **−1,96 e +1,96**: célula dentro do esperado, sem associação local significativa.

## 📊 Visualização dos Resíduos — Heatmap Interativo

- Gera um heatmap com `plotly` destacando células com resíduo **> 1,96** em roxo
- Células dentro do intervalo esperado aparecem em branco
- O arquivo `resid_pisa.html` é salvo para visualização externa

In [ ]:
fig = go.Figure()

maxz = np.max(tab_cont.standardized_resids) + 0.1
minz = np.min(tab_cont.standardized_resids) - 0.1

colorscale = ['purple' if i > 1.96 else '#FAF9F6' for i in np.arange(minz, maxz, 0.01)]

fig.add_trace(
    go.Heatmap(
        x=tab_cont.standardized_resids.columns,
        y=tab_cont.standardized_resids.index,
        z=np.array(tab_cont.standardized_resids),
        text=tab_cont.standardized_resids.values,
        texttemplate='%{text:.2f}',
        showscale=False,
        colorscale=colorscale))

fig.update_layout(
    title='<b>Resíduos Padronizados Ajustados</b>',
    title_x=0.5,
    height=600,
    width=600)

fig.write_html('resid_pisa.html')
fig.show()

---
# ✅ Conclusão

Esta análise utilizou a **PCA** para resumir o desempenho educacional dos países em um único fator latente, e o **teste qui-quadrado + resíduos padronizados** para investigar sua associação com o grupo dos países.

| Etapa | Técnica | Objetivo |
|-------|---------|----------|
| 1 | **PCA** | Extrair fator de desempenho educacional geral (PISA 2022) |
| 2 | **Qui-quadrado** | Testar associação entre o fator categorizado e o grupo do país |
| 3 | **Resíduos Padronizados** | Identificar células com associação local significativa |

> **Nota:** Como a variável `group` possui apenas 2 categorias, não é possível gerar um mapa perceptual bidimensional (ANACOR).  
> Neste caso, a análise encerra-se nos resíduos padronizados ajustados.  
> Caso existissem mais variáveis categóricas, poderia ser realizada uma **ACM**.